# chain-rule-elementwise — ex2: write tanh_back and softplus_back from the elementwise chain rule

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `chain-rule-elementwise`. Running the final beacon cell reports progress against the `Backprop: Elementwise chain rule` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Elementwise chain rule` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`chain-rule-elementwise`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "chain-rule-elementwise"
DD_SUBTOPIC = "Backprop: Elementwise chain rule"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Elementwise chain rule — quick refresher

For elementwise `out = f(x)`, the local Jacobian is diagonal and the chain rule collapses to `dL/dx[i] = dL/dout[i] * f'(x[i])`.

**Worked exemplar.** `out = tanh(x)`:
```
tanh'(x) = 1 - tanh(x)**2 = 1 - out**2     # use the cached out
tanh_back(grad_out, out, x) = grad_out * (1 - out**2)
```
Reusing `out` skips the recomputation of `tanh(x)` during backward.

### Exercise 2 — write tanh_back and softplus_back from the elementwise chain rule

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the elementwise chain rule to derive tanh_back (uses cached out) and softplus_back (derivative is sigmoid(x)), returning grad_in = grad_out * f'(x) per-position.
> Keywords: tanh, softplus, chain-rule, elementwise-derivative
> ```

**KCs targeted:** `chain-rule-elementwise`, `back-fn-uses-cached-out`

Implement TWO elementwise back fns. Both have signature `(grad_out, out, x) -> grad_in` with shape `grad_in.shape == x.shape`.

**1. `tanh_back(grad_out, out, x) -> grad_in`** — for `out = tanh(x)`.
   Math: `tanh'(x) = 1 - tanh(x)**2 = 1 - out**2`.
   Return `grad_out * (1 - out ** 2)`. Reuse the cached `out`.

**2. `softplus_back(grad_out, out, x) -> grad_in`** — for `out = softplus(x) = log(1 + exp(x))`.
   Math: `softplus'(x) = sigmoid(x) = 1 / (1 + exp(-x))`.
   Return `grad_out * t.sigmoid(x)`. Note: depends on `x`, NOT `out` — softplus' derivative happens to be a different elementary function.

The test checks shape preservation, non-unit grad_out scaling, the limit behavior (`tanh_back` vanishes as `|x| -> inf`; `softplus_back` saturates to 0 for large negative `x` and to 1 for large positive `x`), and cross-checks against `torch.autograd`.

In [ ]:
def tanh_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    """Gradient of out = tanh(x). Use the cached `out`."""
    raise NotImplementedError()


def softplus_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    """Gradient of out = softplus(x). Derivative is sigmoid(x)."""
    raise NotImplementedError()


def _test_ex2():
    # --- tanh_back ---
    x = t.tensor([-2.0, -0.5, 0.0, 0.5, 2.0])
    out = t.tanh(x)
    g = tanh_back(t.ones(5), out, x)
    expected = 1 - out ** 2
    assert g.shape == x.shape, f'tanh_back shape: {g.shape}'
    assert t.allclose(g, expected), f'tanh_back value: {g} vs {expected}'

    # tanh_back chain-rule scaling with non-unit grad_out.
    grad_out = t.tensor([3.0, -1.0, 2.0, 0.5, -4.0])
    g = tanh_back(grad_out, out, x)
    assert t.allclose(g, grad_out * (1 - out ** 2)), 'tanh_back chain rule failed'

    # Saturation: tanh_back ~ 0 at |x|=10.
    x_sat = t.tensor([-10.0, 10.0])
    out_sat = t.tanh(x_sat)
    g_sat = tanh_back(t.ones(2), out_sat, x_sat)
    assert t.all(g_sat.abs() < 1e-4), f'tanh_back must saturate to ~0 at |x|=10: {g_sat}'

    # --- softplus_back ---
    x = t.tensor([-2.0, -0.5, 0.0, 0.5, 2.0])
    out = t.nn.functional.softplus(x)
    g = softplus_back(t.ones(5), out, x)
    expected = t.sigmoid(x)
    assert g.shape == x.shape, f'softplus_back shape: {g.shape}'
    assert t.allclose(g, expected), f'softplus_back value: {g} vs {expected}'

    # softplus_back at x=0 = sigmoid(0) = 0.5 EXACTLY.
    x0 = t.tensor([0.0, 0.0])
    g0 = softplus_back(t.ones(2), t.nn.functional.softplus(x0), x0)
    assert t.allclose(g0, t.full((2,), 0.5)), f'softplus_back(0) must be 0.5: {g0}'

    # Saturation: softplus_back -> 0 at very negative x, -> 1 at very positive x.
    x_lim = t.tensor([-10.0, 10.0])
    out_lim = t.nn.functional.softplus(x_lim)
    g_lim = softplus_back(t.ones(2), out_lim, x_lim)
    assert g_lim[0] < 1e-4, f'softplus_back(-10) must be ~0: {g_lim[0]}'
    assert g_lim[1] > 1 - 1e-4, f'softplus_back(10) must be ~1: {g_lim[1]}'

    # --- cross-check vs autograd ---
    xa = t.tensor([-1.0, 0.5, 2.0], requires_grad=True)
    (t.tanh(xa).sum()).backward()
    g_ref_tanh = xa.grad.clone()
    xa.grad = None
    (t.nn.functional.softplus(xa).sum()).backward()
    g_ref_sp = xa.grad.clone()

    x_det = xa.detach()
    g_tanh = tanh_back(t.ones(3), t.tanh(x_det), x_det)
    g_sp   = softplus_back(t.ones(3), t.nn.functional.softplus(x_det), x_det)
    assert t.allclose(g_tanh, g_ref_tanh, atol=1e-5), f'tanh vs autograd: {g_tanh} vs {g_ref_tanh}'
    assert t.allclose(g_sp,   g_ref_sp,   atol=1e-5), f'softplus vs autograd: {g_sp} vs {g_ref_sp}'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def tanh_back(grad_out, out, x):
    return grad_out * (1 - out ** 2)


def softplus_back(grad_out, out, x):
    return grad_out * t.sigmoid(x)
```

**Two cache strategies, one signature.** `tanh_back` uses the cached `out` because `tanh'(x) = 1 - tanh(x)**2` is naturally written in terms of the output. `softplus_back` uses `x` because softplus' derivative IS sigmoid — a different elementary function that isn't expressible cleanly via `out = log(1 + exp(x))`.

**Why `softplus_back` ignores its `out` argument.** The uniform back-fn signature `(grad_out, out, x)` always passes `out` so the dispatcher doesn't have to know which back fn needs it. Unused arguments are normal — the back fn just doesn't reference them.

**Saturation matters.** `tanh_back` vanishes at the tails, which is the source of the classic vanishing-gradient problem with deep tanh networks. `softplus_back` saturates to 1 on the positive side (like ReLU) and to 0 on the negative side, but smoothly — this is the appeal of softplus as a smooth ReLU substitute.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()